In [1]:
import pandas as pd
df =  pd.read_csv('customer_shopping_behavior.csv')

In [2]:
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [3]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [4]:
df['Review Rating'] = df.groupby(['Category'])['Review Rating'].transform( lambda x : x.fillna(x.median() ))

In [5]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [6]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns = {'purchase_amount_(usd)':'purchase_amount'})

In [7]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [8]:
labels = ['Younge Adult','Adult','Middle_Aged','Senior']
df['age_group'] = pd.qcut(df['age'],q=4,labels=labels)

In [9]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle_Aged
1,19,Younge Adult
2,50,Middle_Aged
3,21,Younge Adult
4,45,Middle_Aged
5,46,Middle_Aged
6,63,Senior
7,27,Younge Adult
8,26,Younge Adult
9,57,Middle_Aged


In [10]:
#create column perchase_frequency_days
mapping_frequency = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly':30,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Annually': 365,
    'Every 3 Months':90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(mapping_frequency)
    

In [11]:
df[['purchase_frequency_days','frequency_of_purchases']].head(20)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [12]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [13]:
print((df['discount_applied']==df['promo_code_used']).all())
df=df.drop('promo_code_used', axis=1)

True


In [14]:
!pip install psycopg2-binary sqlalchemy

In [25]:
# step 1 connect to postgrey sql
# print(psycopg2.__version__)
import psycopg2
from sqlalchemy import create_engine

username = 'postgres'
password = '********'
host = 'localhost'
port = '5432'
database = 'customer_behavior'
engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

# step 2 load dataframe in to postgre sql
table_name='customers'
df.to_sql(table_name,engine, if_exists='replace',index=False)
print('Data successfully loaded .....')


Data successfully loaded .....
